# 10 - Optimized Retrieval v2 Evaluation

Evaluates a law-aware dense retrieval variant with query expansion. This does not use benchmark labels as hints. It uses a hand-written Turkish legal keyword/synonym map to infer likely laws and search inside those laws.

In [ ]:
from pathlib import Path
import sys
import importlib

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    print('Not running in Google Colab; using local filesystem paths.')

DRIVE_ROOT = Path('/content/drive/MyDrive/rag')
sys.path = [str(DRIVE_ROOT)] + [p for p in sys.path if p != str(DRIVE_ROOT)]
for name in list(sys.modules):
    if name == 'src' or name.startswith('src.'):
        del sys.modules[name]
importlib.invalidate_caches()

benchmark_csv = DRIVE_ROOT / 'data/benchmark/gold_benchmark_v1.csv'
index_root = DRIVE_ROOT / 'indexes/official_law_v3'
output_dir = DRIVE_ROOT / 'outputs/retrieval_eval'

for path in [benchmark_csv, index_root / 'index_manifest.json', index_root / 'dense/embeddings.npy']:
    if not path.exists():
        raise FileNotFoundError(path)

benchmark_csv, index_root

In [ ]:
import importlib.util
import subprocess
import sys

required_modules = {
    'sentence_transformers': 'sentence-transformers',
    'faiss': 'faiss-cpu',
    'rank_bm25': 'rank-bm25',
    'tqdm': 'tqdm',
}

missing_packages = [package for module, package in required_modules.items() if importlib.util.find_spec(module) is None]
if missing_packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages])
else:
    print('All optimized retrieval dependencies are already installed.')

In [ ]:
import torch
from src.evaluation_optimized_retrieval import evaluate_optimized_retrieval

device = 'cuda' if torch.cuda.is_available() else 'cpu'
summary = evaluate_optimized_retrieval(
    benchmark_csv=benchmark_csv,
    index_root=index_root,
    output_predictions_csv=output_dir / 'optimized_retrieval_v2_predictions.csv',
    output_summary_json=output_dir / 'optimized_retrieval_v2_summary.json',
    top_k=30,
    device=device,
)

summary

In [ ]:
import pandas as pd

optimized = summary['metrics']
baseline = pd.read_csv(output_dir / 'retrieval_weight_sweep_summary_v1.csv', dtype=str, keep_default_na=False)
dense = baseline[baseline['mode'].eq('dense')].iloc[0].to_dict()

rows = [
    {'mode': 'dense_baseline', **{k: float(dense[k]) for k in ['doc_hit@5', 'doc_hit@10', 'article_hit@5', 'article_hit@10', 'article_hit@30', 'article_mrr', 'article_ndcg@5']}},
    {'mode': 'optimized_v2', **{k: optimized[k] for k in ['doc_hit@5', 'doc_hit@10', 'article_hit@5', 'article_hit@10', 'article_hit@30', 'article_mrr', 'article_ndcg@5']}},
]
compare_df = pd.DataFrame(rows)
compare_path = output_dir / 'optimized_retrieval_v2_comparison.csv'
compare_df.to_csv(compare_path, index=False, encoding='utf-8-sig')
compare_df

In [ ]:
pred = pd.read_csv(output_dir / 'optimized_retrieval_v2_predictions.csv', dtype=str, keep_default_na=False)
print('law_filter_rate:', summary['law_filter_rate'])
pred[['question_id', 'topic', 'law_filters', 'gold_article_keys', 'retrieved_citations_top30']].head(15)